# Labwork 4 — Architecture: SOLID, DRY, KISS *(cleaned starter)*

Each exercise gives code that **violates one principle**. Refactor the file in place so the principle holds,
then check with `!python` and `!mypy --strict`. Solutions are in the `exercise*.py` correction files.

Exercises 3, 4 and 5 have you write interfaces. Follow the two conventions from Lecture 4: name each one with a leading capital `I` (`ICoffeeShop`, `IDeliverable`, ...), and keep it **pure** -- every method decorated `@abstractmethod`, with `...` for a body, no `__init__` and no attributes. If a class of yours needs to hold state or share code, it is an abstract base class, not an interface: leave it its plain name.

Run once to create the working directory.

In [ ]:
import os, sys

# The interpreter that is running this notebook.
# On macOS there is often no `python` command at all, only `python3`, and adding a
# shell alias does not help: `!` cells run a NON-interactive shell that never reads
# your profile, and `%%script` launches the program directly, without any shell.
# `sys.executable` is an absolute path, so it keeps working after a `cd` too.
PY = sys.executable
os.makedirs('code', exist_ok=True)
print('code/ ready')
print(f'interpreter: {PY}')


## Exercise 1 — SRP (Single Responsibility)
The class below mixes two responsibilities (identity and address). Split it so each class has one reason to change.

In [ ]:
%%writefile code/exercise1.py
class CoffeeShop:
    def __init__(self, name: str, city: str, zip_code: int) -> None:
        self.__name = name
        self.__city = city
        self.__zip_code = zip_code

    @property
    def name(self) -> str:
        return self.__name

    @property
    def city(self) -> str:
        return self.__city

    @property
    def zip_code(self) -> int:
        return self.__zip_code

    def change_address(self, city: str, zip_code: int) -> None:
        self.__city = city
        self.__zip_code = zip_code


if __name__ == '__main__':
    cs = CoffeeShop('La Viet', 'Ha Noi', 111000)
    cs.change_address('Neuville de Poitou', 86170)
    print(f'CoffeeShop name is "{cs.name}"')


In [ ]:
!{PY} code/exercise1.py
!{PY} -m mypy --strict code/exercise1.py

## Exercise 2 — OCP (Open/Closed)
`InvoiceService` must be edited every time a new company is added. Refactor with polymorphism so a new company
needs **no change** to existing code.

In [ ]:
%%writefile code/exercise2.py
class Company:
    def __init__(self, name: str) -> None:
        self.__name = name

    @property
    def name(self) -> str:
        return self.__name


class CompanyA(Company): pass
class CompanyB(Company): pass
class CompanyC(Company): pass
class CompanyD(Company): pass


class InvoiceService:
    @staticmethod
    def generate_invoice(company: Company) -> str:
        if isinstance(company, CompanyA):
            return "some format of invoice for A company"
        if isinstance(company, CompanyB):
            return "some format of invoice for B company"
        if isinstance(company, CompanyC):
            return "some format of invoice for C company"
        return "error"


if __name__ == "__main__":
    for company in (CompanyA('A'), CompanyB('B'), CompanyC('C'), CompanyD('D')):
        print(InvoiceService.generate_invoice(company))


In [ ]:
!{PY} code/exercise2.py
!{PY} -m mypy --strict code/exercise2.py

## Exercise 3 — LSP (Liskov Substitution)
`B` breaks the contract of `CoffeeShop.takeaway` (it raises instead of honouring it). Refactor so every
`CoffeeShop` is substitutable — e.g. move the delivery capability into its own interface.

In [ ]:
%%writefile code/exercise3.py
class CoffeeShop:
    def __init__(self, name: str) -> None:
        self.__name = name

    @property
    def name(self) -> str:
        return self.__name

    def takeaway(self) -> str:
        return "Delivery soon..."


class A(CoffeeShop):
    def takeaway(self) -> str:
        return "Delivery at most 30 minutes"


class B(CoffeeShop):
    def takeaway(self) -> str:
        raise Exception('We do not have takeaway service')   # <-- LSP violation


if __name__ == "__main__":
    def display(shop: CoffeeShop) -> None:
        print(f'CoffeeShop {shop.name}: {shop.takeaway()}')

    display(A('A'))
    display(B('B'))   # blows up: B is not substitutable for CoffeeShop


In [ ]:
!{PY} code/exercise3.py
!{PY} -m mypy --strict code/exercise3.py

## Exercise 4 — ISP (Interface Segregation)
`ICoffeeShop` is too fat: classes are forced to `raise NotImplementedError`. Split it into segregated interfaces.

In [ ]:
%%writefile code/exercise4.py
from abc import ABC, abstractmethod


class ICoffeeShop(ABC):
    # traditional shops
    @abstractmethod
    def brew_by_espresso_machine(self) -> None: ...
    @abstractmethod
    def brew_machine_pour_over(self) -> None: ...
    # third wave shops
    @abstractmethod
    def brew_by_hand_held_espresso_maker(self) -> None: ...
    @abstractmethod
    def brew_manual_pour_over(self) -> None: ...
    # both
    @abstractmethod
    def brew_filter_coffee(self) -> None: ...


class Traditional(ICoffeeShop):
    def brew_by_espresso_machine(self) -> None:
        print("brewing by espresso machine")

    def brew_machine_pour_over(self) -> None:
        print("brewing machine pour over")

    def brew_filter_coffee(self) -> None:
        print("brewing filter coffee")

    def brew_by_hand_held_espresso_maker(self) -> None:
        raise NotImplementedError("We don't brew by hand held espresso maker")

    def brew_manual_pour_over(self) -> None:
        raise NotImplementedError("We don't brew manual pour over")


if __name__ == "__main__":
    Traditional().brew_filter_coffee()


In [ ]:
!{PY} code/exercise4.py
!{PY} -m mypy --strict code/exercise4.py

## Exercise 5 — DIP (Dependency Inversion)
`Delivery` depends on the concrete `Customer` and `CoffeeShop`. Refactor so it depends on abstractions (interfaces).

In [ ]:
%%writefile code/exercise5.py
class CoffeeShop:
    def __init__(self, name: str) -> None:
        self.__name = name

    def get_payment(self) -> None:
        print(f"{self.__name} gets the payment")

    def deliver_coffee(self) -> None:
        print(f"{self.__name} delivers the coffee")


class Customer:
    def __init__(self, name: str) -> None:
        self.__name = name

    def make_payment(self) -> None:
        print(f"{self.__name} makes the payment")

    def receive_coffee(self) -> None:
        print(f"{self.__name} receives the coffee")


class Delivery:
    def __init__(self, customer: Customer, coffee_shop: CoffeeShop) -> None:
        self.__customer = customer
        self.__coffee_shop = coffee_shop

    def delivers(self) -> None:
        self.__customer.make_payment()
        self.__coffee_shop.get_payment()
        self.__coffee_shop.deliver_coffee()
        self.__customer.receive_coffee()


if __name__ == '__main__':
    Delivery(Customer('Uncle Bob'), CoffeeShop('La Viet')).delivers()


In [ ]:
!{PY} code/exercise5.py
!{PY} -m mypy --strict code/exercise5.py

*Reminder: KISS (keep it simple) and DRY (don't repeat yourself) apply throughout; and **YAGNI** — you aren't gonna need it — keep interfaces only as wide as today's need.*